B20b. Аблация организатора: три модератора при фиксированном составе.....


In [1]:
import sys, os, re, json
from itertools import combinations

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/panel_mechanisms')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
import statsmodels.api as sm
BASE = 'experiments/clinical_protocols/delphi_panels/'
REL = {'gpt': PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv', 'qwen': BASE + 'heterogeneous_qwen_organizer_complete/llm_as_judge/delphi_base_noroles_judge.csv', 'deepseekv4': BASE + 'heterogeneous_deepseek_organizer_complete/llm_as_judge/delphi_base_noroles_judge.csv'}
NAME = {'gpt': 'организатор gpt-oss-120b', 'qwen': 'организатор qwen3-235b', 'deepseekv4': 'организатор deepseek-v4-flash'}
D = {k: rd(v) for k, v in REL.items()}
A = load_arms(verbose=False)
M = meta()
ITEMS_ALL, POS = load_items(verbose=False)
print('плеч аблации: %d, кейсов в каждом: %s' % (len(D), {k: len(v) for k, v in D.items()}))


плеч аблации: 3, кейсов в каждом: {'gpt': 229, 'qwen': 229, 'deepseekv4': 229}


In [2]:
rows = []
for k, d in D.items():
    ps = participants(d)
    wmu = d['winner_model_uri'].value_counts()
    cc = [c for c in d.columns if re.match('r[123]_completion_tokens__', c)]
    v = d[cc].apply(pd.to_numeric, errors='coerce')
    org_cols = [c for c in ['organizer_after_r1_completion_tokens', 'organizer_after_r2_completion_tokens', 'final_synthesis_completion_tokens'] if c in d.columns]
    ov = d[org_cols].apply(pd.to_numeric, errors='coerce')
    lab = pd.DataFrame({p: d['r1_emergency__' + p].map(norm_label) for p in ps})
    rows.append({'организатор': k, 'строк': len(d), 'winner_model_uri': list(wmu.index)[0].split('/')[-2] if len(wmu) else '?', 'единственный winner': len(wmu) == 1, 'участники': ', '.join((p.replace('_default', '') for p in ps)), 'вызовов участников': int(v.notna().sum().sum()), 'на потолке (участники)': int((v >= 10000).sum().sum()), 'вызовов организатора': int(ov.notna().sum().sum()), 'на потолке (организатор)': int((ov >= 10000).sum().sum()), 'полнота голосов R1': float(lab.notna().all(axis=1).mean())})
CH = pd.DataFrame(rows).set_index('организатор')
show(CH, 'Проверки целостности')
print()
print('ВЫВОД ПО ЦЕЛОСТНОСТИ. Манипуляция состоялась: winner_model_uri в каждом')
print('плече ровно один и совпадает с заявленным организатором, состав')
print('участников идентичен, число строк везде 229.')
print()
tot_trunc = int(CH['на потолке (участники)'].sum() + CH['на потолке (организатор)'].sum())
print('Обрезание: всего %d вызовов на потолке из ~%d — это 0.1-0.9%% у одного' % (tot_trunc, int(CH['вызовов участников'].sum() + CH['вызовов организатора'].sum())))
print('плеча. Для сравнения, поломка, из-за которой пересчитывался весь проект,')
print('давала 17-39%% обрезанных вызовов. Здесь эффект такого масштаба')
print('невозможен, но в шаге 10 мы всё равно проверим устойчивость с')
print('исключением затронутых кейсов.')



Проверки целостности
------------------------------------------------------------
             строк     winner_model_uri  единственный winner                  участники  вызовов участников  на потолке (участники)  вызовов организатора  на потолке (организатор)  \
организатор                                                                                                                                                                           
gpt            229         gpt-oss-120b                 True  deepseekv4, gpt120b, qwen                1155                       0                   687                         0   
qwen           229  qwen3-235b-a22b-fp8                 True  deepseekv4, gpt120b, qwen                1137                       0                   687                         0   
deepseekv4     229    deepseek-v4-flash                 True  deepseekv4, gpt120b, qwen                1140                       1                   687                         3   

 

In [3]:
S = pd.DataFrame({k: pd.to_numeric(v['Judge_score'], errors='coerce') for k, v in D.items()}).dropna()
single = {m: float(js(A, 'baseline ' + m).mean()) for m in ['gpt', 'qwen', 'deepseekv4']}
res = pd.DataFrame({'Judge панели': S.mean(), 'SD': S.std(ddof=1), 'SE': [se_clt(S[c]) for c in S.columns], 'сила организатора в одиночку': pd.Series(single)})
res['прирост панели над своим организатором'] = res['Judge панели'] - res['сила организатора в одиночку']
res.index = [NAME[i] for i in res.index]
show(res.sort_values('Judge панели', ascending=False), 'Три организатора, один состав')
print()
print('размах между лучшим и худшим организатором: %.4f' % (S.mean().max() - S.mean().min()))
print()
print('ОРИЕНТИРЫ ИЗ ПРЕЖНИХ НОУТБУКОВ, чтобы понимать масштаб:')
print('   вся премия гетерогенности (P-base − homog gpt)  = +0.031  [A1a]')
print('   минимальная обнаружимая разница при n = 229     =  0.027  [A4]')
print('   эффект дизайна ролей (randomroles − noroles)    = +0.023  [A4]')
print('   головной эффект панели (P-base − baseline gpt)  = +0.164  [A1a]')



Три организатора, один состав
------------------------------------------------------------
                               Judge панели     SD     SE  сила организатора в одиночку  прирост панели над своим организатором
организатор deepseek-v4-flash        0.8111 0.1222 0.0081                        0.6401                                  0.1710
организатор gpt-oss-120b             0.7899 0.1278 0.0084                        0.6258                                  0.1641
организатор qwen3-235b               0.7592 0.1402 0.0093                        0.5622                                  0.1970

размах между лучшим и худшим организатором: 0.0519

ОРИЕНТИРЫ ИЗ ПРЕЖНИХ НОУТБУКОВ, чтобы понимать масштаб:
   вся премия гетерогенности (P-base − homog gpt)  = +0.031  [A1a]
   минимальная обнаружимая разница при n = 229     =  0.027  [A4]
   эффект дизайна ролей (randomroles − noroles)    = +0.023  [A4]
   головной эффект панели (P-base − baseline gpt)  = +0.164  [A1a]


In [4]:
rows = []
for a, b in combinations(S.columns, 2):
    d = S[a] - S[b]
    ct = cluster_test(d, M['Sub Clinical Specialties'].reindex(d.index))
    g = M['Sub Clinical Specialties'].reindex(d.index)
    gm = d.groupby(g).mean()
    se_dom = float(np.sqrt(se_clt(d) ** 2 + gm.var(ddof=1) / gm.size))
    rows.append({'контраст': '%s − %s' % (a, b), 'n': len(d), 'Δ': float(d.mean()), 'SE наив.': se_clt(d), 'SE кластер': ct['se_cluster'], 'SE + домен': se_dom, 'p Уилкоксон': float(stats.wilcoxon(d)[1]), 'p кластер': ct['p'], 'CI низ (домен)': d.mean() - 1.96 * se_dom, 'CI верх (домен)': d.mean() + 1.96 * se_dom})
C = pd.DataFrame(rows)
C['p Холм (семья из 3)'] = holm(C['p Уилкоксон'])
C['вердикт'] = np.where(C['p Холм (семья из 3)'] < 0.05, 'значим', 'не значим')
show(C.set_index('контраст'), 'Попарные контрасты организаторов')



Попарные контрасты организаторов
------------------------------------------------------------
                     n       Δ  SE наив.  SE кластер  SE + домен  p Уилкоксон  p кластер  CI низ (домен)  CI верх (домен)  p Холм (семья из 3) вердикт
контраст                                                                                                                                              
gpt − qwen         229  0.0308    0.0077      0.0090      0.0116       0.0001     0.0030          0.0080           0.0535               0.0003  значим
gpt − deepseekv4   229 -0.0211    0.0078      0.0077      0.0141       0.0133     0.0134         -0.0487           0.0065               0.0133  значим
qwen − deepseekv4  229 -0.0519    0.0078      0.0066      0.0110       0.0000     0.0000         -0.0734          -0.0303               0.0000  значим


In [5]:
rk = pd.DataFrame({'Judge панели': S.mean(), 'сила в одиночку': pd.Series(single)})
rk['ранг панели'] = rk['Judge панели'].rank(ascending=False)
rk['ранг силы'] = rk['сила в одиночку'].rank(ascending=False)
show(rk.sort_values('Judge панели', ascending=False), 'Порядок организаторов против порядка их одиночной силы')
rho, pv = stats.spearmanr(rk['Judge панели'], rk['сила в одиночку'])
print()
print('ранговая корреляция: rho = %+.3f (p = %s при n = 3)' % (rho, fmt_p(pv)))
print()
print('ОСТОРОЖНО. При трёх точках совпадение порядков случается с вероятностью')
print('1/6 = 0.167 просто так. Совпадение НЕ является доказательством, это')
print('лишь согласующееся наблюдение. Настоящая проверка требует больше')
print('организаторов — см. шаг 11.')



Порядок организаторов против порядка их одиночной силы
------------------------------------------------------------
            Judge панели  сила в одиночку  ранг панели  ранг силы
deepseekv4        0.8111           0.6401       1.0000     1.0000
gpt               0.7899           0.6258       2.0000     2.0000
qwen              0.7592           0.5622       3.0000     3.0000

ранговая корреляция: rho = +1.000 (p = 0.0e+00 при n = 3)

ОСТОРОЖНО. При трёх точках совпадение порядков случается с вероятностью
1/6 = 0.167 просто так. Совпадение НЕ является доказательством, это
лишь согласующееся наблюдение. Настоящая проверка требует больше
организаторов — см. шаг 11.


In [6]:
MEMB = ['baseline gpt', 'baseline qwen', 'baseline deepseekv4']
ITEMS = {k: item_frame(v) for k, v in D.items()}
common = set(POS)
for f in ITEMS.values():
    common &= set(f)
KEYS = sorted(common)
print('пунктов, общих всем трём плечам и базовому набору: %d' % len(KEYS))
base_maj, base_union, pts = ({}, {}, {})
for k in KEYS:
    v = [int(ITEMS_ALL[m][k]['met']) for m in MEMB]
    base_maj[k] = int(sum(v) >= 2)
    base_union[k] = int(sum(v) >= 1)
    pts[k] = ITEMS_ALL[MEMB[0]][k]['points']

def per_case(fn):
    rows = []
    for k in KEYS:
        rows.append({'case': k[0], 'val': fn(k), 'points': pts[k]})
    d = pd.DataFrame(rows)
    return d.groupby('case').apply(lambda g: (g['val'] * g['points']).sum() / g['points'].sum(), include_groups=False)
maj_s = per_case(lambda k: base_maj[k])
uni_s = per_case(lambda k: base_union[k])
rows = []
for org, f in ITEMS.items():
    pan_s = per_case(lambda k, f=f: int(f[k]['met']))
    ix = pan_s.index.intersection(maj_s.index)
    rows.append({'организатор': org, 'панель (по пунктам)': float(pan_s.loc[ix].mean()), 'большинство состава': float(maj_s.loc[ix].mean()), 'объединение (оракул)': float(uni_s.loc[ix].mean()), 'вклад синтеза': float((pan_s.loc[ix] - maj_s.loc[ix]).mean()), 'превышение оракула': float((pan_s.loc[ix] - uni_s.loc[ix]).mean())})
LIFT = pd.DataFrame(rows).set_index('организатор')
show(LIFT, 'Вклад организатора: панель против голосования состава')
print()
print('Колонка «большинство состава» одинакова во всех строках — так и должно')
print('быть, состав ведь не менялся. Это встроенная проверка, что мы')
print('действительно сравниваем организаторов, а не что-то ещё.')
print()
print('размах вклада синтеза между организаторами: %.4f' % (LIFT['вклад синтеза'].max() - LIFT['вклад синтеза'].min()))


пунктов, общих всем трём плечам и базовому набору: 2249



Вклад организатора: панель против голосования состава
------------------------------------------------------------
             панель (по пунктам)  большинство состава  объединение (оракул)  вклад синтеза  превышение оракула
организатор                                                                                                   
gpt                       0.7900               0.6295                0.7688         0.1605              0.0211
qwen                      0.7620               0.6295                0.7688         0.1325             -0.0069
deepseekv4                0.8141               0.6295                0.7688         0.1846              0.0452

Колонка «большинство состава» одинакова во всех строках — так и должно
быть, состав ведь не менялся. Это встроенная проверка, что мы
действительно сравниваем организаторов, а не что-то ещё.

размах вклада синтеза между организаторами: 0.0521


In [7]:
rows = []
liftser = {}
for org, f in ITEMS.items():
    pan_s = per_case(lambda k, f=f: int(f[k]['met']))
    ix = pan_s.index.intersection(maj_s.index)
    liftser[org] = pan_s.loc[ix] - maj_s.loc[ix]
for a, b in combinations(liftser, 2):
    d = (liftser[a] - liftser[b]).dropna()
    rows.append({'контраст': '%s − %s' % (a, b), 'n': len(d), 'Δ вклада': float(d.mean()), 'SE': se_clt(d), 'p': float(stats.wilcoxon(d)[1])})
LC = pd.DataFrame(rows)
LC['p Холм'] = holm(LC['p'])
show(LC.set_index('контраст'), 'Различается ли вклад синтеза между организаторами')



Различается ли вклад синтеза между организаторами
------------------------------------------------------------
                     n  Δ вклада     SE      p  p Холм
контраст                                              
gpt − qwen         220    0.0280 0.0078 0.0006  0.0012
gpt − deepseekv4   220   -0.0241 0.0077 0.0051  0.0051
qwen − deepseekv4  220   -0.0521 0.0076 0.0000  0.0000


In [8]:
AX_ORDER = ['accuracy', 'completeness', 'reasoning', 'urgency', 'uncertainty', 'communication_quality']
axis_of = {k: ITEMS_ALL[MEMB[0]][k]['axis'] for k in KEYS}
W = pd.Series({ax: sum((pts[k] for k in KEYS if axis_of[k] == ax)) for ax in AX_ORDER})
W = W / W.sum()

def axis_scores(f):
    out = {}
    for ax in AX_ORDER:
        ks = [k for k in KEYS if axis_of[k] == ax]
        num = sum((int(f[k]['met']) * pts[k] for k in ks))
        den = sum((pts[k] for k in ks))
        out[ax] = num / den if den else np.nan
    return pd.Series(out)
AS = pd.DataFrame({org: axis_scores(f) for org, f in ITEMS.items()}).T
show(AS[AX_ORDER], 'Доля выполненных пунктов по осям рубрики')
print()
dec = pd.DataFrame({'вес оси': W, 'ds − qwen (разн.)': AS.loc['deepseekv4'] - AS.loc['qwen'], 'gpt − qwen (разн.)': AS.loc['gpt'] - AS.loc['qwen']})
dec['вклад ds−qwen'] = dec['ds − qwen (разн.)'] * dec['вес оси']
dec['доля вклада, %'] = 100 * dec['вклад ds−qwen'] / dec['вклад ds−qwen'].sum()
show(dec.loc[AX_ORDER].sort_values('вклад ds−qwen', ascending=False), 'Разложение разрыва «лучший минус худший организатор» по осям')
print()
print('Для сравнения, из C28: у контраста «панель против одиночной модели»')
print('62%% прироста давали completeness и uncertainty, а accuracy лишь 9.4%%.')
print('Если у организаторов профиль другой, значит они различаются не тем же,')
print('чем панель отличается от одиночной модели.')



Доля выполненных пунктов по осям рубрики
------------------------------------------------------------
            accuracy  completeness  reasoning  urgency  uncertainty  communication_quality
gpt           0.8903        0.7228     0.8991   0.8963       0.5567                 0.6429
qwen          0.8925        0.6408     0.8618   0.8893       0.5671                 0.6490
deepseekv4    0.9027        0.7134     0.9352   0.9405       0.6729                 0.6786


Разложение разрыва «лучший минус худший организатор» по осям
------------------------------------------------------------
                       вес оси  ds − qwen (разн.)  gpt − qwen (разн.)  вклад ds−qwen  доля вклада, %
completeness            0.3201             0.0726              0.0819         0.0232         43.8462
uncertainty             0.0983             0.1058             -0.0104         0.0104         19.6154
reasoning               0.1111             0.0734              0.0373         0.0082         15.3846
urgen

In [9]:
rows = []
for org, f in ITEMS.items():
    em = lo = pick = 0
    for k in KEYS:
        v = sum((int(ITEMS_ALL[m][k]['met']) for m in MEMB))
        pn = int(f[k]['met'])
        if pn == 1 and v == 0:
            em += 1
        elif pn == 0 and v >= 2:
            lo += 1
        elif pn == 1 and v == 1:
            pick += 1
    rows.append({'организатор': org, 'пунктов': len(KEYS), 'эмерджентных': em, 'потерянных': lo, 'подхвачено у меньшинства': pick, 'нетто': em - lo})
EM = pd.DataFrame(rows).set_index('организатор')
show(EM, 'Что организатор добавляет и что теряет')
print()
print('«подхвачено у меньшинства» — пункт выполнил ровно один участник, и')
print('организатор его сохранил. Различие здесь показывает, насколько')
print('организатор готов идти против большинства своей панели.')



Что организатор добавляет и что теряет
------------------------------------------------------------
             пунктов  эмерджентных  потерянных  подхвачено у меньшинства  нетто
организатор                                                                    
gpt             2249           246          69                       236    177
qwen            2249           214          85                       211    129
deepseekv4      2249           275          57                       255    218

«подхвачено у меньшинства» — пункт выполнил ровно один участник, и
организатор его сохранил. Различие здесь показывает, насколько
организатор готов идти против большинства своей панели.


In [10]:
rows = []
for org, d in D.items():
    t = true_label(d).map(ORD)
    p = pred_triage(d).map(ORD)
    ok = t.notna() & p.notna()
    e = (p - t)[ok]
    rows.append({'организатор': org, 'n': int(ok.sum()), 'accuracy': float((p[ok] == t[ok]).mean()), 'недотриаж': float((p[ok] < t[ok]).mean()), 'перетриаж': float((p[ok] > t[ok]).mean()), 'знаковая ошибка': float(e.mean()), 'средняя |ошибка|': float(e.abs().mean())})
TR = pd.DataFrame(rows).set_index('организатор')
show(TR, 'Триаж по организаторам')
print()
print('размах accuracy: %.4f   размах рубричного балла: %.4f' % (TR['accuracy'].max() - TR['accuracy'].min(), S.mean().max() - S.mean().min()))
pt = {}
for org, d in D.items():
    t = true_label(d).map(ORD)
    p = pred_triage(d).map(ORD)
    pt[org] = (p == t).astype(float).where(t.notna() & p.notna())
PT = pd.DataFrame(pt).dropna()
for a, b in combinations(PT.columns, 2):
    m = mcnemar(PT[a].astype(bool), PT[b].astype(bool))
    print('   McNemar %-12s vs %-12s: n01=%d n10=%d p=%s' % (a, b, m['n01'], m['n10'], fmt_p(m['p'])))
print()
print('Если разрыв по триажу мал и McNemar незначим, предсказание B22')
print('подтверждается: организатор управляет ТЕКСТОМ, но не МЕТКОЙ.')



Триаж по организаторам
------------------------------------------------------------
               n  accuracy  недотриаж  перетриаж  знаковая ошибка  средняя |ошибка|
организатор                                                                        
gpt          229    0.7860     0.0131     0.2009           0.1878            0.2140
qwen         229    0.8297     0.0044     0.1659           0.1616            0.1703
deepseekv4   229    0.8253     0.0087     0.1659           0.1572            0.1747

размах accuracy: 0.0437   размах рубричного балла: 0.0519
   McNemar gpt          vs qwen        : n01=15 n10=5 p=0.041
   McNemar gpt          vs deepseekv4  : n01=13 n10=4 p=0.049
   McNemar qwen         vs deepseekv4  : n01=6 n10=7 p=1.000

Если разрыв по триажу мал и McNemar незначим, предсказание B22
подтверждается: организатор управляет ТЕКСТОМ, но не МЕТКОЙ.


In [11]:
rows = []
for org, f in ITEMS.items():
    r = {'организатор': org}
    for m in MEMB:
        mm = m.replace('baseline ', '')
        agree = np.mean([int(f[k]['met']) == int(ITEMS_ALL[m][k]['met']) for k in KEYS])
        r['совпадение с ' + mm] = float(agree)
    r['ближайший участник'] = max(MEMB, key=lambda m: np.mean([int(f[k]['met']) == int(ITEMS_ALL[m][k]['met']) for k in KEYS])).replace('baseline ', '')
    r['совпадает ли с собой'] = r['ближайший участник'] == org
    rows.append(r)
DOM = pd.DataFrame(rows).set_index('организатор')
show(DOM, 'С чьими одиночными вердиктами совпадает итоговый протокол')
print()
print('ЧИТАЕТСЯ ТАК. Если бы работало самопредпочтение, в колонке «ближайший')
print('участник» стоял бы сам организатор во всех трёх строках. Если же')
print('ближайшим оказывается одна и та же модель независимо от того, кто')
print('организатор, значит протокол тяготеет к сильнейшему участнику, а не к')
print('тому, кто держит перо.')



С чьими одиночными вердиктами совпадает итоговый протокол
------------------------------------------------------------
             совпадение с gpt  совпадение с qwen  совпадение с deepseekv4 ближайший участник  совпадает ли с собой
организатор                                                                                                       
gpt                    0.7430             0.7065                   0.7590         deepseekv4                 False
qwen                   0.7479             0.7381                   0.7675         deepseekv4                 False
deepseekv4             0.7261             0.6932                   0.7519         deepseekv4                  True

ЧИТАЕТСЯ ТАК. Если бы работало самопредпочтение, в колонке «ближайший
участник» стоял бы сам организатор во всех трёх строках. Если же
ближайшим оказывается одна и та же модель независимо от того, кто
организатор, значит протокол тяготеет к сильнейшему участнику, а не к
тому, кто держит перо.


In [12]:
rows = []
for org, d in D.items():
    ps = participants(d)
    rr = pd.to_numeric(d.get('delphi_rounds_executed'), errors='coerce')
    lab = {}
    for r in (1, 2, 3):
        for p in ps:
            c = 'r%d_emergency__%s' % (r, p)
            lab[r, p] = d[c].map(norm_label) if c in d.columns else pd.Series(index=d.index, dtype=object)
    orgp = [p for p in ps if p.startswith(org)]
    orgp = orgp[0] if orgp else None
    if orgp is None:
        continue
    moved_to_org = moved = 0
    for p in ps:
        if p == orgp:
            continue
        l1 = lab[1, p]
        last = lab[3, p].where(lab[3, p].notna(), lab[2, p])
        last = last.where(last.notna(), l1)
        o1 = lab[1, orgp]
        ch = l1.notna() & last.notna() & (l1 != last) & (rr > 1)
        moved += int(ch.sum())
        moved_to_org += int((ch & (last == o1)).sum())
    rows.append({'организатор': org, 'смен мнения у прочих': moved, 'из них к позиции организатора': moved_to_org, 'доля': moved_to_org / moved if moved else np.nan})
MV = pd.DataFrame(rows).set_index('организатор')
show(MV, 'Куда движутся участники, когда меняют мнение')
print()
print('При трёх участниках случайное движение «к позиции организатора»')
print('составляло бы примерно 50%%, поскольку альтернатив у метки немного.')
print('Существенное превышение указывало бы на доминирование организатора.')



Куда движутся участники, когда меняют мнение
------------------------------------------------------------
             смен мнения у прочих  из них к позиции организатора   доля
организатор                                                            
gpt                            42                             36 0.8571
qwen                           47                             40 0.8511
deepseekv4                     56                             51 0.9107

При трёх участниках случайное движение «к позиции организатора»
составляло бы примерно 50%%, поскольку альтернатив у метки немного.
Существенное превышение указывало бы на доминирование организатора.


In [13]:
rows = []
for org, d in D.items():
    rr = pd.to_numeric(d.get('delphi_rounds_executed'), errors='coerce')
    tot = [c for c in d.columns if re.match('r[123]_total_tokens__', c)]
    orgc = [c for c in ['organizer_after_r1_total_tokens', 'organizer_after_r2_total_tokens', 'final_synthesis_total_tokens'] if c in d.columns]
    gen = d[tot].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
    agg = d[orgc].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
    cmp_cols = [c.replace('total', 'completion') for c in tot + orgc if c.replace('total', 'completion') in d.columns]
    cmpl = d[cmp_cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
    w = d['winner_protocol'].map(words) if 'winner_protocol' in d.columns else pd.Series(dtype=float)
    unan = pd.DataFrame({p: d['r1_emergency__' + p].map(norm_label) for p in participants(d)})
    rows.append({'организатор': org, 'Judge': float(pd.to_numeric(d['Judge_score'], errors='coerce').mean()), 'ср. раундов': float(rr.mean()), 'стоп после R1': float((rr == 1).mean()), 'единогласие R1': float((unan.dropna().nunique(axis=1) == 1).mean()), 'S генерация': float(gen.mean()), 'V агрегация': float(agg.mean()), 'total': float((gen + agg).mean()), 'completion': float(cmpl.mean()), 'слов в протоколе': float(w.mean()) if len(w) else np.nan})
B = pd.DataFrame(rows).set_index('организатор')
B['Judge на 10k total'] = B['Judge'] / (B['total'] / 10000.0)
show(B, 'Динамика протокола и бюджет по организаторам')
print()
print('ВАЖНО ДЛЯ §2. Из A5 известно, что запас паритета над USC@7 составлял')
print('всего 3%% по total-токенам. Если организаторы различаются по стоимости')
print('сильно, выбор организатора становится ещё и бюджетным решением.')



Динамика протокола и бюджет по организаторам
------------------------------------------------------------
             Judge  ср. раундов  стоп после R1  единогласие R1  S генерация  V агрегация      total  completion  слов в протоколе  Judge на 10k total
организатор                                                                                                                                          
gpt         0.7899       1.6812         0.5240          0.7181   14537.0131   12209.4410 26746.4541  11962.0830          478.5371              0.2953
qwen        0.7592       1.6550         0.5066          0.7118   14586.4367   10925.9782 25512.4148  10496.5502          468.4236              0.2976
deepseekv4  0.8111       1.6594         0.5109          0.7156   14508.5415   14574.3319 29082.8734  14686.5546          604.7555              0.2789

ВАЖНО ДЛЯ §2. Из A5 известно, что запас паритета над USC@7 составлял
всего 3%% по total-токенам. Если организаторы различаются по стоимости
си

In [14]:
parts = []
for org, d in D.items():
    s = pd.to_numeric(d['Judge_score'], errors='coerce')
    w = d['winner_protocol'].map(words)
    ix = s.dropna().index.intersection(w.dropna().index)
    parts.append(pd.DataFrame({'y': s.loc[ix].to_numpy(), 'w': w.loc[ix].to_numpy(), 'org': org, 'case': ix}))
L = pd.concat(parts, ignore_index=True)
L['lw'] = np.log(L['w'].clip(lower=1))
Dm = pd.get_dummies(L['org'], prefix='o', drop_first=True).astype(float)
g = L['case'].astype('category').cat.codes
m0 = sm.OLS(L['y'], sm.add_constant(Dm)).fit(cov_type='cluster', cov_kwds={'groups': g})
m1 = sm.OLS(L['y'], sm.add_constant(pd.concat([Dm, L[['lw']]], axis=1))).fit(cov_type='cluster', cov_kwds={'groups': g})
out = pd.DataFrame({'без контроля длины': m0.params, 'p': m0.pvalues, 'с контролем длины': m1.params, 'p ': m1.pvalues})
show(out, 'Эффект организатора до и после контроля длины протокола')
print()
print('Если коэффициенты при организаторе почти не меняются после добавления')
print('log(длины), эффект организатора не сводится к многословию.')



Эффект организатора до и после контроля длины протокола
------------------------------------------------------------
        без контроля длины      p  с контролем длины     p 
const               0.8111 0.0000            -0.0645 0.7746
lw                     NaN    NaN             0.1370 0.0001
o_gpt              -0.0211 0.0066             0.0114 0.2988
o_qwen             -0.0519 0.0000            -0.0167 0.1393

Если коэффициенты при организаторе почти не меняются после добавления
log(длины), эффект организатора не сводится к многословию.


In [15]:
bad = set()
for org, d in D.items():
    cc = [c for c in d.columns if re.match('r[123]_completion_tokens__', c)] + [c for c in ['organizer_after_r1_completion_tokens', 'organizer_after_r2_completion_tokens'] if c in d.columns]
    v = d[cc].apply(pd.to_numeric, errors='coerce')
    bad |= set(v.index[(v >= 10000).any(axis=1)])
print('кейсов с хотя бы одним обрезанным вызовом в любом плече: %d' % len(bad))
S2 = S.drop(index=[i for i in bad if i in S.index])
cmp_ = pd.DataFrame({'все кейсы': S.mean(), 'без обрезанных': S2.mean()})
cmp_['сдвиг'] = cmp_['без обрезанных'] - cmp_['все кейсы']
show(cmp_, 'Устойчивость к исключению обрезанных кейсов (n = %d)' % len(S2))


кейсов с хотя бы одним обрезанным вызовом в любом плече: 4

Устойчивость к исключению обрезанных кейсов (n = 225)
------------------------------------------------------------
            все кейсы  без обрезанных   сдвиг
gpt            0.7899          0.7890 -0.0010
qwen           0.7592          0.7600  0.0008
deepseekv4     0.8111          0.8129  0.0018


In [16]:
save(CH.reset_index(), 'organizer_ablation_fixed_panel_integrity.csv')
save(res.reset_index(), 'organizer_ablation_fixed_panel_headline.csv')
save(C, 'organizer_ablation_fixed_panel_contrasts.csv')
save(LIFT.reset_index(), 'organizer_ablation_fixed_panel_synthesis_lift.csv')
save(AS.reset_index(), 'organizer_ablation_fixed_panel_axes.csv')
save(EM.reset_index(), 'organizer_ablation_fixed_panel_emergent.csv')
save(TR.reset_index(), 'organizer_ablation_fixed_panel_triage.csv')
save(DOM.reset_index(), 'organizer_ablation_fixed_panel_dominance.csv')
save(B.reset_index(), 'organizer_ablation_fixed_panel_budget.csv')


сохранено: out/organizer_ablation_fixed_panel_integrity.csv  (3 строк)


сохранено: out/organizer_ablation_fixed_panel_headline.csv  (3 строк)
сохранено: out/organizer_ablation_fixed_panel_contrasts.csv  (3 строк)
сохранено: out/organizer_ablation_fixed_panel_synthesis_lift.csv  (3 строк)
сохранено: out/organizer_ablation_fixed_panel_axes.csv  (3 строк)
сохранено: out/organizer_ablation_fixed_panel_emergent.csv  (3 строк)
сохранено: out/organizer_ablation_fixed_panel_triage.csv  (3 строк)
сохранено: out/organizer_ablation_fixed_panel_dominance.csv  (3 строк)
сохранено: out/organizer_ablation_fixed_panel_budget.csv  (3 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\panel_mechanisms\\results\\organizer_ablation_fixed_panel_budget.csv'